# 13-4. 공통 스키마·시간 정규화·보고서 예제

## Goal

- 원문 시각과 UTC 시각을 함께 보존합니다.
- 아티팩트별 시각 의미를 구분합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

표준 라이브러리만 사용하며 합성 레코드를 변환합니다.


## Steps

### 시간대가 명시된 시각 정규화

시간대가 없는 값은 추측하지 않고 오류로 격리합니다.


In [1]:
from datetime import datetime, timezone


def to_utc(value: str) -> str | None:
    if value == "":
        return None
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        raise ValueError("시간대가 없는 시각은 변환할 수 없습니다")
    return parsed.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def normalize_record(raw: dict) -> dict:
    return {
        "case_id": raw["case_id"],
        "host": raw["host"],
        "artifact": raw["artifact"],
        "timestamp_original": raw["timestamp"],
        "timestamp_utc": to_utc(raw["timestamp"]),
        "timestamp_kind": raw["timestamp_kind"],
        "source_file": raw["source_file"],
    }


record = normalize_record({
    "case_id": "SYNTHETIC", "host": "HOST-A", "artifact": "evtx",
    "timestamp": "2026-09-01T09:06:00+09:00", "timestamp_kind": "event_created",
    "source_file": "events.csv",
})
print(record)


{'case_id': 'SYNTHETIC', 'host': 'HOST-A', 'artifact': 'evtx', 'timestamp_original': '2026-09-01T09:06:00+09:00', 'timestamp_utc': '2026-09-01T00:06:00Z', 'timestamp_kind': 'event_created', 'source_file': 'events.csv'}


## Checks

UTC 변환 결과와 원문·의미 필드 보존을 확인합니다.


In [2]:
assert record["timestamp_utc"] == "2026-09-01T00:06:00Z"
assert record["timestamp_original"].endswith("+09:00")
try:
    to_utc("2026-09-01 09:06:00")
except ValueError:
    print("시간대 없는 시각 거부 확인")


시간대 없는 시각 거부 확인


## Next Steps

보고서에서는 화면 표시 한도와 전체 JSONL 위치를 함께 안내합니다.
